In [11]:
import numpy as np
from hmmlearn.hmm import GaussianHMM
from hmmlearn.hmm import CategoricalHMM
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import pandas as pd

In [12]:
data = pd.read_pickle('BG_021_allsession_data.pkl')

data = data[data['change_sizes_TF'] != 1]
data = data[data['outcomes'].isin(['Hit', 'Miss', 'FA'])]

In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from hmmlearn.hmm import CategoricalHMM
from sklearn.preprocessing import LabelEncoder
from matplotlib.backends.backend_pdf import PdfPages

# Define fixed colors for hidden states
HIDDEN_STATE_COLORS = {0: "blue", 1: "red", 2: "green"}  # Consistent coloring across plots

# Group outcomes by (mouse_id, session_date)
unique_combinations = data.groupby(['mouse_id', 'session_date'])['outcomes'].apply(list)

# Flatten all outcomes and encode them into integers
encoder = LabelEncoder()
all_outcomes_flat = [outcome for session in unique_combinations for outcome in session]
encoded_outcomes = encoder.fit_transform(all_outcomes_flat)

# Convert each session's outcomes into integer-encoded NumPy arrays
session_arrays = []
start_idx = 0
for outcomes in unique_combinations:
    session_length = len(outcomes)
    session_arrays.append(np.array(encoded_outcomes[start_idx : start_idx + session_length]))
    start_idx += session_length  # Move index forward

# Concatenate all sessions for training
train_data = np.concatenate(session_arrays).reshape(-1, 1)

# Hyperparameter search: Train multiple models and keep the best one
best_model = None
best_score = -np.inf  

for _ in range(1000):  # Train multiple times
    model = CategoricalHMM(n_components=3, n_features=len(encoder.classes_), random_state=None, n_iter=100)
    model.fit(train_data)
    log_prob = model.score(train_data)  
    
    if log_prob > best_score:  
        best_score = log_prob
        best_model = model

print("Best Model Score:", best_score)

# Print estimated parameters
print("Estimated Start Probabilities:\n", best_model.startprob_)
print("Estimated Transition Matrix:\n", best_model.transmat_)
print("Estimated Emission Probabilities:\n", best_model.emissionprob_)

# Save figures into a multi-page PDF
pdf_filename = "hidden_states_plots.pdf"
with PdfPages(pdf_filename) as pdf:
    # Decode each session separately using the best model
    for (mouse_id, session_date), session_outcomes in zip(unique_combinations.index, session_arrays):
        _, hidden_states = best_model.decode(session_outcomes.reshape(-1, 1), algorithm="viterbi")

        # Create DataFrame for plotting
        df = pd.DataFrame({
            "Trial": np.arange(len(session_outcomes)),
            "Observation": encoder.inverse_transform(session_outcomes.ravel()),  # Convert back to categorical labels
            "Hidden State": hidden_states
        })

        # Ensure correct outcome order
        outcome_order = ['FA', 'Hit', 'Miss']
        df["Observation"] = pd.Categorical(df["Observation"], categories=outcome_order, ordered=True)

        # Create figure
        fig, ax = plt.subplots(figsize=(10, 5))
        sns.swarmplot(
            x="Trial", 
            y="Observation", 
            data=df, 
            hue="Hidden State", 
            palette=HIDDEN_STATE_COLORS, 
            size=3, 
            ax=ax
        )
        sns.despine()

        # Customize plot
        plt.title(f"Observed Outcomes vs. Inferred Hidden States\nMouse {mouse_id}, Session {session_date}")
        plt.xlabel("Trial")
        plt.ylabel("Observed Outcome")
        plt.legend(title="Hidden State")

        # Save current figure to the PDF
        pdf.savefig(fig)
        plt.close(fig)  # Close the figure to free memory

print(f"Plots saved to {pdf_filename}")
